# 🔆 8장 · 내일의 발전량: 회귀 분석 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 8장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 8-1, 8-2, 8-4, 8-5, 8-6, 8-7 |
| ☆ 선택 | 자습이나 심화 | 8-3, 8-8, 8-9, 원자료 정리 |

**데이터**
- 골턴의 가족 키(1886년): GitHub 저장소 `vincentarelbundock/Rdatasets`에서 바로 읽습니다.
- 서울 태양광 발전량: 한국전력거래소 "지역별 시간별 태양광 및 풍력 발전량" 2025년(출처 한국전력거래소, 공공데이터포털, 공공누리 제1유형).
- 서울 날씨: 기상청 종관기상관측(ASOS) 서울(108) 시간 자료 2023-01-01~2026-10-17(출처 기상청, 기상자료개방포털, 공공누리 제1유형).
- 미니 프로젝트: 한국전력거래소 "시간별 전국 전력수요량" 2021~2025년(출처 한국전력거래소, 공공데이터포털, 공공누리 제1유형).

한국 데이터 세 파일은 **수업 자료로 받은 정리본(UTF-8 CSV)**을 씁니다. 코랩 왼쪽 파일 창에 올려 두면 되고, 다른 폴더에 두었다면 [파이썬 실습 8-1]의 `DATA = ""` 한 줄만 그 폴더로 고칩니다(예: `DATA = "/content/drive/MyDrive/data/"`). 포털에서 원자료를 직접 받아 정리하는 ☆ 셀은 8-9 뒤에 있고, 원자료가 없으면 건너뜁니다.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**와 나라별 공휴일 목록 **holidays**(미니 프로젝트)를 설치합니다. 코랩에 이미 있으면 바로 넘어갑니다. 판다스, 스태츠모델, 사이킷런, 맷플롯립은 코랩에 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib holidays

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch08/data/"
for f in [
    "asos_서울108_시간자료_전체열_2023_2026.csv",
    "kpx_시간별_전국_전력수요량_2021_2025.csv",
    "kpx_지역별_시간별_태양광풍력_발전량_2025.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 키에서 서울의 발전량까지

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 골턴의 키 데이터는 인터넷 주소에서 바로 읽습니다. 한국 데이터 세 파일은 수업 자료로 받은 정리본을 코랩 왼쪽 파일 창에 올려 두고 시작합니다. 파일을 다른 폴더에 두었다면 [파이썬 실습 8-1]의 `DATA` 한 줄만 고치면 됩니다. 회귀에는 스태츠모델을 씁니다. 계수, p값, 신뢰구간을 통계학 교과서처럼 보여 주기 때문입니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 8-1 ★] 골턴의 키와 평균으로의 회귀

골턴 가족 키 데이터에서 아들과 아버지의 키를 센티미터로 바꾸고, 키 큰 아버지와 키 작은 아버지의 아들 키를 견주시오. 회귀선도 구해 그리시오.

① 데이터를 읽고 인치를 cm로 바꿉니다. ② 위아래 10%를 골라 평균을 견줍니다. ③ 회귀선을 구해 45° 대각선과 함께 그립니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
DATA = ""      # 수업 자료로 받은 한국 데이터 CSV를 둔 폴더 (코랩 파일 창에 바로 올렸으면 빈칸 그대로)

URL = ("https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/"
       "master/csv/HistData/GaltonFamilies.csv")
galton = pd.read_csv(URL)
sons = galton[galton["gender"] == "male"].copy()
sons["아버지"] = sons["father"] * 2.54            # 인치 → cm
sons["아들"] = sons["childHeight"] * 2.54
print("자녀", len(galton), "명 가운데 아들", len(sons), "명")
print(sons[["아버지", "아들"]].agg(["mean", "std"]).round(2).to_string())

tall = sons[sons["아버지"] >= sons["아버지"].quantile(0.9)]     # 키 큰 아버지 상위 10%
short = sons[sons["아버지"] <= sons["아버지"].quantile(0.1)]    # 키 작은 아버지 하위 10%
tall_son = sons[sons["아들"] >= sons["아들"].quantile(0.9)]    # 거꾸로: 키 큰 아들 상위 10%
print(f"키 큰 아버지 {len(tall)}명: {tall['아버지'].mean():.1f}cm → 아들 {tall['아들'].mean():.1f}cm")
print(f"키 작은 아버지 {len(short)}명: {short['아버지'].mean():.1f}cm → 아들 {short['아들'].mean():.1f}cm")
print(f"키 큰 아들 {len(tall_son)}명: {tall_son['아들'].mean():.1f}cm → 그 아버지 {tall_son['아버지'].mean():.1f}cm")

b, a = np.polyfit(sons["아버지"], sons["아들"], 1)   # 기울기, 절편
r = sons["아버지"].corr(sons["아들"])
print(f"회귀선: 아들 = {a:.1f} + {b:.3f} × 아버지,  상관계수 {r:.3f},  R² {r ** 2:.3f}")

plt.figure(figsize=(5, 4.4))
plt.scatter(sons["아버지"], sons["아들"], s=8, alpha=0.4, color="gray")
xs = np.array([155, 200])
plt.plot(xs, a + b * xs, color="#C8234B", label="회귀선")
plt.plot(xs, xs, "k--", lw=1, label="아들 = 아버지")
plt.xlabel("아버지 키(cm)")
plt.ylabel("아들 키(cm)")
plt.legend()
plt.show()

`quantile(0.9)`는 아래에서 90% 되는 값입니다. 키 큰 아버지의 아들은 5.3cm, 키 작은 아버지의 아들은 5.7cm만큼 아버지보다 평균 쪽에 있습니다. 거꾸로 골라도 마찬가지입니다. `np.polyfit(x, y, 1)`은 최소제곱 직선(1차식)의 기울기와 절편을 차례로 돌려줍니다.

#### [파이썬 실습 8-2 ★] 최소제곱 직선을 손 계산처럼 구하기

[예제 8-1]의 계산을 코드로 따라 하고, 기울기를 바꾸면 잔차 제곱합이 어떻게 변하는지 그리시오.

In [ ]:
study = pd.DataFrame({"공부시간": [2, 4, 6, 8, 10], "점수": [60, 70, 75, 85, 90]})
x, y = study["공부시간"], study["점수"]
sxy = ((x - x.mean()) * (y - y.mean())).sum()     # 편차 곱의 합
sxx = ((x - x.mean()) ** 2).sum()                 # x 편차 제곱의 합
w = sxy / sxx
b = y.mean() - w * x.mean()
print(f"기울기 w = {sxy} ÷ {sxx} = {w},  절편 b = {b}")

pred = b + w * x
sse = ((y - pred) ** 2).sum()
sst = ((y - y.mean()) ** 2).sum()
print("잔차:", (y - pred).tolist(), " 잔차의 합:", (y - pred).sum())
print(f"SSE = {sse},  SST = {sst},  R² = {1 - sse / sst:.3f}")

ws = np.linspace(2.5, 5.0, 51)                    # 기울기를 바꿔 가며 SSE 재기
sses = [((y - (y.mean() - v * x.mean() + v * x)) ** 2).sum() for v in ws]
for v in [3.0, 3.5, 3.75, 4.0, 4.5]:
    print(f"w = {v:4} → SSE = {((y - (y.mean() - v * x.mean() + v * x)) ** 2).sum():5.1f}")
plt.figure(figsize=(5, 3.2))
plt.plot(ws, sses, color="gray")
plt.scatter([w], [sse], color="#C8234B", zorder=3)
plt.xlabel("기울기 w")
plt.ylabel("잔차 제곱합(SSE)")
plt.show()

손으로 구한 기울기 3.75, 절편 53.5, R² 0.987이 그대로 나옵니다. 잔차의 합은 0입니다. 기울기를 3.75에서 어느 쪽으로 바꿔도 잔차 제곱합이 커집니다. 절편은 늘 ȳ − w·x̄로 맞추어 직선이 평균점을 지나게 했습니다.

#### [파이썬 실습 8-3 ☆] 경사하강법으로 같은 직선 찾기

w = 0, b = 0에서 출발해 학습률 0.01로 3,000걸음 경사하강법을 하고, 공식의 답과 견주시오.

In [ ]:
xa, ya = x.to_numpy(float), y.to_numpy(float)
w_gd, b_gd, lr = 0.0, 0.0, 0.01                   # 0에서 출발, 한 걸음 크기(학습률) 0.01
losses = []
for step in range(3000):
    err = (b_gd + w_gd * xa) - ya                 # 예측 − 실제
    losses.append((err ** 2).mean())
    if step in (0, 100, 1000):
        print(f"{step:4d}걸음: w = {w_gd:6.3f}, b = {b_gd:6.3f}, 손실 = {losses[-1]:8.2f}")
    w_gd -= lr * 2 * (err * xa).mean()            # 손실이 줄어드는 쪽으로 w 이동
    b_gd -= lr * 2 * err.mean()                   # 손실이 줄어드는 쪽으로 b 이동
print(f"3000걸음: w = {w_gd:.3f}, b = {b_gd:.3f}, 손실 = {losses[-1]:.2f}")

plt.figure(figsize=(5, 3.2))
plt.plot(losses, color="#C8234B")
plt.yscale("log")
plt.xlabel("걸음 수")
plt.ylabel("손실(로그 눈금)")
plt.show()

처음 손실 5,890은 100걸음 만에 242로, 1,000걸음 뒤에는 1.89로 줄었습니다. 3,000걸음 뒤에는 공식의 답에 도착했습니다. 100걸음에서 w가 8.73까지 지나쳤다가 되돌아오는 것도 보입니다. 학습률을 0.05로 키우면 골짜기를 넘어 튕겨 나가니 직접 바꿔 보십시오.

#### [파이썬 실습 8-4 ★] 서울 태양광과 날씨 기록 잇기

한국전력거래소 파일에서 서울 태양광을 고르고, 기상청 서울 관측 자료와 시각으로 이으시오. 시각을 한 칸씩 밀어 보며 맞물림을 확인하고, 일사 빈칸을 세시오.

① 거래일과 거래시간으로 "끝나는 시각"을 만듭니다. ② 날씨를 같은 시각에 붙입니다. ③ 빈칸이 밤인지, 측정이 빠진 것인지 가립니다.

In [ ]:
# 수업 자료의 한국 데이터 세 파일이 DATA 폴더에 없으면, 이 셀이 올릴 창을 엽니다(코랩).
import os
FILES = ['kpx_지역별_시간별_태양광풍력_발전량_2025.csv', 'asos_서울108_시간자료_전체열_2023_2026.csv', 'kpx_시간별_전국_전력수요량_2021_2025.csv']
missing = [f for f in FILES if not os.path.exists(DATA + f)]
if missing:
    print('없는 파일:', missing)
    from google.colab import files
    files.upload()
else:
    print('세 파일이 모두 있습니다.')

In [ ]:
SOLAR = "kpx_지역별_시간별_태양광풍력_발전량_2025.csv"   # 수업 자료로 받은 정리본
ASOS = "asos_서울108_시간자료_전체열_2023_2026.csv"

kpx = pd.read_csv(DATA + SOLAR)
print("한국전력거래소 파일:", kpx.shape, "/ 지역", kpx["지역"].nunique(), "곳")
seoul = kpx[(kpx["지역"] == "서울시") & (kpx["연료원"] == "태양광")].copy()
seoul["일시"] = pd.to_datetime(seoul["거래일"]) + pd.to_timedelta(seoul["거래시간"], unit="h")
seoul = seoul.rename(columns={"전력거래량(MWh)": "발전량"})[["일시", "발전량"]]
print(f"서울 태양광: {len(seoul)}시각, 한 해 {seoul['발전량'].sum():,.0f}MWh, 가장 많은 시각 {seoul['발전량'].max():.1f}MWh")

names = {"일시": "일시", "기온(°C)": "기온", "습도(%)": "습도", "일사(MJ/m2)": "일사",
         "전운량(10분위)": "운량", "적설(cm)": "적설"}
wx = pd.read_csv(DATA + ASOS, usecols=list(names)).rename(columns=names)
wx["일시"] = pd.to_datetime(wx["일시"])
print("기상청 서울 관측:", wx.shape, wx["일시"].min(), "~", wx["일시"].max())

for h in [-1, 0, 1]:                               # 시각을 한 칸씩 밀어 보며 잇기
    moved = wx[["일시", "일사"]].assign(일시=wx["일시"] + pd.Timedelta(hours=h))
    m = seoul.merge(moved, on="일시")
    print(f"기상 시각을 {h:+d}시간 밀면 상관계수 {m['발전량'].corr(m['일사'].fillna(0)):.3f}")

df = seoul.merge(wx, on="일시", how="left")        # 같은 시각의 날씨를 옆에 붙인다
df["시각"] = df["일시"].dt.hour
blank = df["일사"].isna()
noon = blank & df["시각"].between(8, 18)
print("일사 빈칸", blank.sum(), "시각 / 그 가운데 한낮(8~18시)", noon.sum(), "시각:",
      df.loc[noon, "일시"].dt.date.unique())
night = blank & ~noon
print(f"밤인데 발전량이 0보다 큰 시각 {(df.loc[night, '발전량'] > 0).sum()}개, 합계 {df.loc[night, '발전량'].sum():.1f}MWh")

df.groupby("시각")["발전량"].mean().plot.bar(color="gray", figsize=(6, 3))
plt.ylabel("평균 발전량(MWh)")
plt.xlabel("시각(그 시각까지 한 시간)")
plt.xticks(range(0, 24, 3), range(0, 24, 3), rotation=0)
plt.show()

> **[코드 읽기] `pd.to_timedelta( )`와 `merge( )`**
>
> `pd.to_timedelta(seoul["거래시간"], unit="h")`는 13을 "13시간"이라는 시간 길이로 바꿉니다. 날짜에 더하면 "2025-01-01 13:00"이 됩니다. `seoul.merge(wx, on="일시", how="left")`는 두 표에서 일시가 같은 줄을 옆으로 붙입니다. `how="left"`는 왼쪽(발전량) 표의 줄을 모두 남기고, 짝이 없으면 빈칸으로 둡니다.

18개 지역, 두 연료원의 기록이 한 파일에 섞여 있어서 서울 태양광만 골랐습니다. 시각을 밀지 않았을 때 상관계수가 0.971로 가장 높습니다. 일사 빈칸 3,978시각 가운데 한낮의 빈칸 9시각은 모두 9월 13일입니다. 밤인데 거래가 있었던 시각도 1,359개입니다.

#### [파이썬 실습 8-5 ★] 일사 하나로 회귀하고 기준선과 겨루기

해가 있던 시각으로 "발전량 ~ 일사" 회귀를 하시오. 1~10월로 배우고 11~12월로 시험해 세 가지 기준선과 견주고, 랜덤 분할 성적과도 견주시오.

① 회귀식을 맞추고 R²를 봅니다. ② 채점 함수 `score( )`를 만들어 같은 잣대로 잽니다. ③ 시간 분할과 랜덤 분할을 견줍니다.

In [ ]:
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

day = df[df["일사"] > 0].copy()                    # 해가 있는 시각만
print("해가 있는 시각:", len(day))
m1 = smf.ols("발전량 ~ 일사", data=day).fit()
print(f"발전량 = {m1.params['Intercept']:.2f} + {m1.params['일사']:.2f} × 일사,  R² {m1.rsquared:.3f}")
print("밤까지 넣으면 R²", round(smf.ols("발전량 ~ 일사", data=df.fillna({"일사": 0})).fit().rsquared, 3))

CUT = "2025-11-01"                                 # 앞 10달로 배우고 뒤 2달로 시험
train, test = day[day["일시"] < CUT], day[day["일시"] >= CUT]
print("학습", len(train), "시각 / 시험", len(test), "시각")

def score(name, y_true, y_pred, unit="MWh"):     # 같은 채점표로 모델을 견준다
    print(f"{name:10s} MAE {mean_absolute_error(y_true, y_pred):6.2f}{unit}   R² {r2_score(y_true, y_pred):6.3f}")

score("평균으로 찍기", test["발전량"], np.full(len(test), train["발전량"].mean()))
score("시각별 평균", test["발전량"], test["시각"].map(train.groupby("시각")["발전량"].mean()))
yday = df.set_index("일시")["발전량"].shift(24)     # 24시각 전 = 어제 같은 시각
score("어제 같은 시각", test["발전량"], test["일시"].map(yday))
fit = smf.ols("발전량 ~ 일사", data=train).fit()
score("일사 회귀", test["발전량"], fit.predict(test))
print(f"  시험 잔차(실제 − 예측)의 평균 {(test['발전량'] - fit.predict(test)).mean():+.2f}MWh")
r_tr, r_te = train_test_split(day, test_size=0.2, random_state=42)
score("(랜덤 분할)", r_te["발전량"], smf.ols("발전량 ~ 일사", data=r_tr).fit().predict(r_te))

plt.figure(figsize=(5, 4))
plt.scatter(day["일사"], day["발전량"], s=4, alpha=0.3, color="gray")
xs = np.array([0, 3.9])
plt.plot(xs, m1.params["Intercept"] + m1.params["일사"] * xs, color="#C8234B")
plt.xlabel("일사(MJ/m², 한 시간)")
plt.ylabel("서울 태양광 발전량(MWh)")
plt.show()

> **[코드 읽기] 식으로 쓰는 회귀 `"발전량 ~ 일사"`**
>
> `smf.ols("발전량 ~ 일사", data=day)`는 "발전량을 일사로 설명하라"는 식을 글자 그대로 씁니다. `~` 왼쪽이 종속변수, 오른쪽이 독립변수입니다. 독립변수를 더하려면 `"발전량 ~ 일사 + 기온"`처럼 `+`로 잇습니다. `.fit( )`이 최소제곱으로 계수를 구하고, `.params`에 절편과 기울기가, `.rsquared`에 R²가 들어 있습니다.

`shift(24)`는 줄을 24칸 아래로 밀어, 각 시각 옆에 24시간 전 값을 놓습니다. 기준선 가운데 가장 나은 "어제 같은 시각"도 1.94MWh를 틀리는데, 일사 회귀는 0.98MWh입니다. 랜덤 분할 R² 0.920은 시간 분할 0.875보다 좋아 보이지만, 이웃 시각을 미리 본 점수입니다.

#### [파이썬 실습 8-6 ★] 겨울의 기울기와 온도 손실 변수

달마다 따로 직선을 그어 기울기를 견주시오. 온도 손실 변수를 만들어 네 모델(M1~M4)을 1~10월로 배우고 11~12월로 시험하시오.

In [ ]:
day["월"] = day["일시"].dt.month
slope = {mo: smf.ols("발전량 ~ 일사", data=g).fit().params["일사"] for mo, g in day.groupby("월")}
print("달마다 다시 그은 직선의 기울기:")
print(pd.Series(slope).round(2).to_string())

day["온도손실"] = day["일사"] * (day["기온"] - 25)  # 햇빛이 셀 때 25℃보다 더운 만큼
train, test = day[day["일시"] < CUT], day[day["일시"] >= CUT]
forms = {"M1": "발전량 ~ 일사",
         "M2": "발전량 ~ 일사 + 기온 + 운량 + 습도",
         "M3": "발전량 ~ 일사 + 기온 + 운량 + 습도 + 온도손실",
         "M4": "발전량 ~ 일사 + 기온 + 운량 + 습도 + 온도손실 + C(시각)"}
for name, f in forms.items():
    fit = smf.ols(f, data=train).fit()
    tr_mae = mean_absolute_error(train["발전량"], fit.fittedvalues)
    print(f"{name} 학습 MAE {tr_mae:.2f}, R² {fit.rsquared:.3f} | ", end="")
    score("시험", test["발전량"], fit.predict(test))
m3 = smf.ols(forms["M3"], data=train).fit()
print(f"온도손실 계수 {m3.params['온도손실']:.4f} (p값 {m3.pvalues['온도손실']:.1e})")
print(f"기온 1℃마다 일사 1당 발전량이 {-m3.params['온도손실'] / m3.params['일사']:.1%}씩 준다")

plt.figure(figsize=(6, 3))
plt.bar(list(slope), list(slope.values()), color="gray")
plt.xticks(range(1, 13))
plt.xlabel("월")
plt.ylabel("일사 1당 발전량(MWh)")
plt.show()

> **[코드 읽기] `C(시각)`과 더미 변수**
>
> 시각은 숫자이지만, 9시와 10시의 차이가 15시와 16시의 차이와 같다고 볼 까닭이 없습니다. `C(시각)`은 "시각을 숫자가 아니라 범주로 다루라"는 뜻입니다. 스태츠모델은 시각마다 0과 1로 된 열(더미 변수)을 만들어, 시각마다 따로 보정값을 둡니다.

`for mo, g in day.groupby("월")`은 달마다 묶은 표를 하나씩 꺼내 줍니다. 기울기가 1월 5.17, 7월 3.35, 12월 5.33입니다. 온도 손실을 넣은 M3에서 시험 MAE가 0.67MWh로 크게 줄고, 학습과 시험의 차이도 거의 없습니다. 온도 손실 계수 −0.0491을 일사 계수로 나누면 1℃에 1.4%입니다.

#### [파이썬 실습 8-7 ★] 예측구간과 오차의 모양

M3로 한 조건의 예측값, 평균의 95% 신뢰구간, 한 시각의 95% 예측구간을 구하시오. 시험 기간에 예측구간이 실제로 몇 %를 담는지 세고, 오차를 히스토그램으로 그리시오.

In [ ]:
from statsmodels.stats.stattools import durbin_watson

cond = pd.DataFrame({"일사": [2.0], "기온": [10.0], "운량": [2], "습도": [50]})
cond["온도손실"] = cond["일사"] * (cond["기온"] - 25)
pf = m3.get_prediction(cond).summary_frame(alpha=0.05)
print(pf[["mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]].to_string(index=False, float_format="%.2f"))

band = m3.get_prediction(test).summary_frame(alpha=0.05)
inside = (test["발전량"] >= band["obs_ci_lower"]) & (test["발전량"] <= band["obs_ci_upper"])
print(f"시험 {len(test)}시각 가운데 95% 예측구간 안: {inside.mean():.1%}"
      f" (아래로 벗어남 {(test['발전량'] < band['obs_ci_lower']).sum()}, 위로 {(test['발전량'] > band['obs_ci_upper']).sum()})")

res = test["발전량"] - m3.predict(test)
print("잔차 가운데 값", round(res.median(), 2), "/ 가장 크게 틀린 시각:")
worst = test.assign(잔차=res).nsmallest(4, "잔차")
print(worst[["일시", "발전량", "일사", "기온", "적설", "잔차"]].to_string(index=False, float_format="%.2f"))
print("더빈-왓슨(M3 학습 잔차):", round(durbin_watson(m3.resid), 2))

plt.figure(figsize=(5, 3.2))
plt.hist(res, bins=40, color="gray")
plt.axvline(0, color="#C8234B")
plt.xlabel("잔차 = 실제 - 예측(MWh)")
plt.ylabel("시각 수")
plt.show()

`get_prediction( )`의 결과에서 mean은 예측값, mean_ci는 평균의 신뢰구간, obs_ci는 한 시각의 예측구간입니다. 예측구간의 폭(약 4.1MWh)은 신뢰구간의 폭(약 0.1MWh)의 30배가 넘습니다. 가장 크게 틀린 네 시각은 모두 눈이 쌓인 12월 5일입니다.

#### [파이썬 실습 8-8 ☆] 이슬점을 더하면: 분산팽창계수

일사, 기온, 습도로 회귀한 모델에 이슬점을 더하고, 변수마다 VIF를 구하시오.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

dew = pd.read_csv(DATA + ASOS, usecols=["일시", "이슬점온도(°C)"]).rename(columns={"이슬점온도(°C)": "이슬점"})
dew["일시"] = pd.to_datetime(dew["일시"])
d8 = day.merge(dew, on="일시")
for cols in [["일사", "기온", "습도"], ["일사", "기온", "습도", "이슬점"]]:
    X = sm.add_constant(d8[cols])
    vif = [variance_inflation_factor(X.values, i) for i in range(1, len(cols) + 1)]
    print(" + ".join(cols), "→ VIF", ", ".join(f"{v:.1f}" for v in vif))
m_3 = smf.ols("발전량 ~ 일사 + 기온 + 습도", data=d8).fit()
m_dew = smf.ols("발전량 ~ 일사 + 기온 + 습도 + 이슬점", data=d8).fit()
print(pd.DataFrame({"이슬점 없이": m_3.params, "이슬점 넣고": m_dew.params}).round(3).to_string())
print("R²:", round(m_3.rsquared, 4), "→", round(m_dew.rsquared, 4))

이슬점을 넣기 전에는 VIF가 모두 2보다 작습니다. 넣고 나면 기온, 습도, 이슬점이 서로를 거의 완전히 설명하므로 VIF가 치솟습니다. 기온의 계수는 −0.024에서 −0.053으로 두 배가 넘게 바뀌었는데, R²는 그대로입니다. 같은 정보를 세 변수가 나누어 갖는 비율만 달라진 것입니다. `sm.add_constant( )`는 절편 열을 붙여 주는데, VIF는 절편을 뺀 변수에 대해서만 봅니다.

#### [파이썬 실습 8-9 ☆] 릿지, 라소, 랜덤 포레스트와 견주기

M3의 변수에 시각을 더해 선형 회귀, 릿지, 라소, 랜덤 포레스트의 시험 성적을 견주시오.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor

feats = ["일사", "기온", "운량", "습도", "온도손실", "시각"]
train, test = day[day["일시"] < CUT], day[day["일시"] >= CUT]
models = {"선형": make_pipeline(StandardScaler(), LinearRegression()),
          "릿지": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30))),
          "라소": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0))}
for name, mdl in models.items():
    mdl.fit(train[feats], train["발전량"])
    score(name, test["발전량"], mdl.predict(test[feats]))
print("라소의 계수:", pd.Series(models["라소"][-1].coef_, index=feats).round(3).to_dict())
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1)
rf.fit(train[feats], train["발전량"])
score("랜덤 포레스트", test["발전량"], rf.predict(test[feats]))
score("  (학습 성적)", train["발전량"], rf.predict(train[feats]))

릿지와 라소는 계수 크기에 벌점을 주므로 `make_pipeline( )`으로 "표준화 → 모델"을 묶어 단위를 먼저 맞춥니다. 세 선형 모델의 성적은 같았고, 라소도 0으로 만든 계수가 없었습니다. 교차검증이 고른 벌점이 작아서 보통 회귀와 거의 같아진 것으로 보입니다. 그래도 "시각"의 계수는 0.054로 아주 작습니다. 랜덤 포레스트는 시험 MAE 0.47MWh로 가장 좋지만, 학습(0.35MWh)과 시험의 차이가 더 큽니다. 랜덤 포레스트는 10장에서 자세히 배웁니다.

> **[더 알아보기] 원자료를 직접 받아 정리하기**
>
> 수업 자료의 세 파일은 공공데이터포털과 기상자료개방포털에서 받은 원자료를 정리한 것입니다. 원자료는 한글 윈도 인코딩(cp949)이라 `encoding="cp949"`로 읽어야 글자가 깨지지 않습니다. 기상청 자료는 한 번에 받을 수 있는 기간이 정해져 있어 여러 파일로 나뉘어 옵니다. 그래서 `pd.concat( )`으로 이어 붙이고, 겹친 시각을 `drop_duplicates("일시")`로 지웁니다. 일사와 강수량의 빈칸은 지우지 않습니다. 밤이거나 비가 안 온 시각이라는 뜻이 있기 때문입니다(3.2). 아래 코드는 원자료 파일이 DATA 폴더에 있을 때만 돌고, 없으면 건너뜁니다.

In [ ]:
# ☆ 원자료에서 정리본 만들기: 공공데이터포털·기상자료개방포털에서 받은 파일이 DATA 폴더에 있을 때만 돈다.
import glob
raw_asos = sorted(glob.glob(DATA + "OBS_ASOS_TIM_*.csv"))      # 기상자료개방포털: 한 번에 1년쯤씩 받은 파일들
if raw_asos:
    parts = [pd.read_csv(f, encoding="cp949") for f in raw_asos]  # 원자료는 한글 윈도 인코딩(cp949)
    asos = pd.concat(parts).drop_duplicates("일시").sort_values("일시")
    asos.to_csv(DATA + "asos_내가정리한_서울108.csv", index=False)  # UTF-8로 저장
    print("ASOS 파일", len(raw_asos), "개 →", len(asos), "시각", asos["일시"].iloc[0], "~", asos["일시"].iloc[-1])
    print("일사 빈칸(밤 포함):", asos["일사(MJ/m2)"].isna().sum(), "/ 강수량 빈칸(비가 안 온 시각):", asos["강수량(mm)"].isna().sum())
else:
    print("기상청 원자료(OBS_ASOS_TIM_*.csv)가 없어 건너뜁니다.")

raw_kpx = sorted(glob.glob(DATA + "한국전력거래소_*태양광*풍력*.csv"))  # 공공데이터포털에서 받은 파일
if raw_kpx:
    sol = pd.read_csv(raw_kpx[0], encoding="cp949")
    sol.columns = ["거래일", "거래시간", "지역", "연료원", "전력거래량(MWh)"]
    sol.to_csv(DATA + "kpx_내가정리한_태양광풍력.csv", index=False)
    print("한국전력거래소 파일:", sol.shape, "/ 거래시간", sol["거래시간"].min(), "~", sol["거래시간"].max())
else:
    print("한국전력거래소 원자료(한국전력거래소_*태양광*풍력*.csv)가 없어 건너뜁니다.")

raw_dem = sorted(glob.glob(DATA + "한국전력거래소_시간별*전력수요량*.csv"))  # 미니 프로젝트: 해마다 한 파일
if raw_dem:
    parts = []
    for f in raw_dem:
        t = pd.read_csv(f, encoding="cp949")
        t.columns = ["날짜"] + [f"{h}시" for h in range(1, 25)]   # 해마다 열 이름을 똑같이
        parts.append(t)
    dem_all = pd.concat(parts).dropna(subset=["날짜"])            # 내용 없는 줄을 버린다
    dem_all = dem_all.drop_duplicates("날짜").sort_values("날짜")
    dem_all.to_csv(DATA + "kpx_내가정리한_전력수요.csv", index=False)
    print("전력수요 파일", len(raw_dem), "개 →", dem_all.shape, dem_all["날짜"].iloc[0], "~", dem_all["날짜"].iloc[-1])
else:
    print("한국전력거래소 원자료(한국전력거래소_시간별*전력수요량*.csv)가 없어 건너뜁니다.")

기상청 파일 네 개를 이으면 33,241시각이 되고, 수업 자료의 정리본과 똑같습니다. 일사 빈칸 14,955시각은 밤이 대부분입니다. 전력수요는 해마다 한 파일씩 받습니다. 열 이름을 `날짜, 1시, …, 24시`로 맞추고, 2025년 파일 끝의 내용 없는 줄을 버리면 1,826일이 됩니다. 내려받은 파일 이름이 다르면 `glob( )` 안의 이름을 고치십시오.

> **핵심 3줄**
>
> 최소제곱 회귀는 오차 제곱의 합이 가장 작은 직선을 찾고, 그 성적은 기준선과 견주어야 뜻이 생깁니다.
>
> 내일을 맞힐 모델은 시간 순서대로 나누어 시험하고, 잔차의 모양과 예측구간으로 언제 크게 틀리는지까지 봅니다.
>
> 일사 회귀의 시험 MAE는 0.98MWh로 "어제 같은 시각"(1.94MWh)의 절반이었고, 온도 손실 변수를 더하자 0.67MWh로 줄었습니다.

## [미니 프로젝트 8] 추우면 난방, 더우면 냉방: 전력 수요의 U자

**질문.** 전국의 하루 전력 수요는 날씨에 얼마나 끌려다닐까요? 추운 날은 난방으로, 더운 날은 냉방으로 전기를 많이 씁니다. 그렇다면 기온과 전력 수요 사이에 직선 하나를 그어도 될까요?

**데이터.** 두 파일을 씁니다. 하나는 한국전력거래소의 "시간별 전국 전력수요량"(2021~2025년)입니다. 한 행이 하루이고, 날짜와 1시~24시의 수요(MW)로 25열입니다. 다른 하나는 03절의 기상청 서울 관측 자료입니다. 서울의 하루 평균 기온을 전국 날씨의 대표로 씁니다. 두 자료가 겹치는 2023~2025년 1,096일이 분석 대상입니다. 수업 자료의 정리본(kpx_시간별_전국_전력수요량_2021_2025.csv)을 DATA 폴더에 두고 시작합니다.


**1단계: 데이터 신원 확인**([파이썬 실습 8-4] 응용). 수요 파일의 빈 줄을 확인하고 하루 수요(GWh)를 만드시오. 서울의 하루 평균 기온을 붙이시오.

In [ ]:
DEMAND = "kpx_시간별_전국_전력수요량_2021_2025.csv"  # 수업 자료로 받은 정리본
dem = pd.read_csv(DATA + DEMAND)
print(dem.shape, "/ 날짜가 빈 줄", dem["날짜"].isna().sum())
dem = dem.dropna(subset=["날짜"])
dem["날짜"] = pd.to_datetime(dem["날짜"])
dem["수요"] = dem.drop(columns="날짜").sum(axis=1) / 1000   # 24시간 MW의 합 → 하루 GWh

temp = pd.read_csv(DATA + ASOS, usecols=["일시", "기온(°C)"])
temp["날짜"] = pd.to_datetime(temp["일시"]).dt.normalize()
temp = temp.groupby("날짜")["기온(°C)"].mean().rename("기온").reset_index()

mp = dem[["날짜", "수요"]].merge(temp, on="날짜")       # 두 자료가 겹치는 날만
print(len(mp), "일:", mp["날짜"].min().date(), "~", mp["날짜"].max().date())
print(mp[["수요", "기온"]].describe().round(1).loc[["mean", "min", "max"]].to_string())

수요 파일 맨 끝에 날짜까지 빈 줄이 하나 있어 버렸습니다. 24시간의 MW를 더해 1,000으로 나누면 하루 GWh입니다. 하루 수요는 1,089~1,997GWh, 기온은 −13.8~32.7℃ 사이입니다.

**2단계: 기준선과 직선**([파이썬 실습 8-5] 응용). 2023~2024년으로 배우고 2025년으로 시험하시오. 평균으로 찍기와 직선 회귀를 견주고, 산점도를 그리시오.

In [ ]:
mp["연도"] = mp["날짜"].dt.year
tr, te = mp[mp["연도"] < 2025], mp[mp["연도"] == 2025]   # 2023~2024년으로 배우고 2025년으로 시험
line = smf.ols("수요 ~ 기온", data=tr).fit()
print(f"직선: 수요 = {line.params['Intercept']:.1f} {line.params['기온']:+.2f} × 기온,  학습 R² {line.rsquared:.3f}")
print(f"상관계수 {mp['수요'].corr(mp['기온']):.3f}")
score("평균으로 찍기", te["수요"], np.full(len(te), tr["수요"].mean()), "GWh")
score("직선", te["수요"], line.predict(te), "GWh")

plt.figure(figsize=(5, 3.6))
plt.scatter(mp["기온"], mp["수요"], s=6, alpha=0.4, color="gray")
plt.xlabel("서울 하루 평균 기온(°C)")
plt.ylabel("전국 하루 전력 수요(GWh)")
plt.show()

상관계수가 0.004, 직선의 학습 R²가 0.002입니다. 직선으로 보면 "기온과 수요는 아무 관계가 없다"입니다. 하지만 산점도는 뚜렷한 U자입니다. 추운 쪽의 오르막과 더운 쪽의 오르막이 직선 하나 안에서 서로 지워진 것입니다. 상관계수가 0이라고 관계가 없는 것은 아닙니다.

**3단계: 제곱항과 쉬는 날 더하기**. 기온의 제곱을 변수로 더해 U자를 그리고, 주말과 공휴일을 뜻하는 "쉬는날" 더미 변수도 더하시오.

In [ ]:
import holidays
kr = holidays.KR(years=[2023, 2024, 2025])
mp["쉬는날"] = ((mp["날짜"].dt.dayofweek >= 5) | mp["날짜"].isin(pd.to_datetime(list(kr)))).astype(int)
tr, te = mp[mp["연도"] < 2025], mp[mp["연도"] == 2025]
u = smf.ols("수요 ~ 기온 + I(기온 ** 2)", data=tr).fit()
u2 = smf.ols("수요 ~ 기온 + I(기온 ** 2) + 쉬는날", data=tr).fit()
print(f"U자: 학습 R² {u.rsquared:.3f}, 수요가 가장 적은 기온 {-u.params['기온'] / (2 * u.params['I(기온 ** 2)']):.1f}℃")
print(f"U자 + 쉬는날: 학습 R² {u2.rsquared:.3f}, 쉬는 날 계수 {u2.params['쉬는날']:.1f}GWh")
score("U자", te["수요"], u.predict(te), "GWh")
score("U자 + 쉬는날", te["수요"], u2.predict(te), "GWh")

t = np.linspace(mp["기온"].min(), mp["기온"].max(), 100)
plt.figure(figsize=(5, 3.6))
plt.scatter(tr["기온"], tr["수요"], s=6, alpha=0.4, color="gray")
plt.plot(t, u.predict(pd.DataFrame({"기온": t})), color="#C8234B")
plt.xlabel("서울 하루 평균 기온(°C)")
plt.ylabel("전국 하루 전력 수요(GWh)")
plt.show()

기온의 제곱을 넣어도 여전히 "직선 회귀"입니다. 계수에 대해 일차식이기 때문입니다. U자 곡선은 2025년 시험 R² 0.446, MAE 112.66GWh로 직선(156.81GWh)보다 훨씬 낫습니다. 쉬는 날을 더하자 MAE가 76.20GWh로 줄었습니다. 쉬는 날에는 같은 기온에서 225.8GWh를 덜 씁니다. 공장과 사무실이 쉬기 때문입니다.

**4단계: 가장 크게 틀린 날 읽기.** 2025년에 예측보다 수요가 가장 적었던 날 다섯을 찾고, 그날이 어떤 날이었는지 쓰시오.

In [ ]:
te = te.assign(잔차=te["수요"] - u2.predict(te))
print(te.nsmallest(5, "잔차")[["날짜", "수요", "기온", "쉬는날", "잔차"]].to_string(index=False, float_format="%.1f"))

다섯 날 가운데 셋이 설 연휴(1월 28~30일)입니다. 주말 하루와 설 연휴 사흘은 같은 "쉬는날"이 아니었습니다. 5월 2일은 근로자의 날과 어린이날 연휴 사이에 낀 금요일, 12월 26일은 연말입니다. 달력에는 평일이지만 많은 사람이 쉰 날입니다.

**5단계: 한계 쓰기.** 이 모델이 아직 모르는 것을 [표 8-3]처럼 세 가지 이상 쓰시오. 12장의 미니 프로젝트에서는 같은 수요를 "어제와 지난주의 수요"로 예측합니다. 기온과 지난주 값 가운데 무엇이 더 쓸모 있을지도 생각해 보십시오.

**제출물.** 그림 2장(기온과 수요의 산점도와 U자 곡선, 잔차가 큰 날 표시), 성적표 1개(평균, 직선, U자, U자 + 쉬는날의 시험 MAE와 R²), 결론 5문장 이내.

**[표 8-4] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 빈 줄, 단위(MW → GWh), 두 자료가 겹치는 기간을 모두 밝힘 | 빈 줄만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 해로 나누고 평균 찍기와 견줌 | 기준선은 있으나 무작위로 나눔 | 기준선 없이 R²만 씀 |
| 기법 적용(30%) | 제곱항과 더미 변수를 넣고 수요가 가장 적은 기온을 구함 | 제곱항만 넣음 | 직선만 쓰거나 코드가 돌지 않음 |
| 해석과 한계(30%) | 상관계수 0의 함정과 연휴의 잔차를 설명하고 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

## ✏️ [연습문제 8-6] 시험 기간 바꾸기

[파이썬 실습 8-6]에서 나누는 날을 2025년 7월 1일로 바꾸어(1~6월로 학습, 7~12월로 시험) M1과 M3의 시험 MAE와 R²를 구하시오. 원래(1~10월 학습, 11~12월 시험)와 견주어 설명하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
CUT2 = "2025-07-01"                                # 앞 6달로 배우고 뒤 6달로 시험
train2, test2 = day[day["일시"] < CUT2], day[day["일시"] >= CUT2]
for name in ["M1", "M3"]:
    fit = smf.ols(forms[name], data=train2).fit()
    score(name, test2["발전량"], fit.predict(test2))